# S6E9: Public Blend Follow-Along | LB 0.94650

This notebook tests public predictions with explicit source attribution. **It does not train a new model.**

Start with `public_notebook` to export the downloaded version-5 output from [Taeyang's Lexsort notebook](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master). **Our submitted file scored 0.94650 public LB**, as reported by the user, improving on the previous 0.94639 submission by **0.00011**. The original author reports 0.94649. Our result uses the downloaded public output; it does not establish that repackaging or tie-breaking caused this difference.

| Submission | Public LB | Evidence |
|---|---:|---|
| Previous locally trained blend | 0.94639 | User-reported |
| Downloaded public Lexsort output | **0.94650** | User-reported after submission |
| Anchor-only and anchor_lexsort modes | Not submitted in this trial | No new measured score |

The 0.94650 result applies to the default `public_notebook` mode. Alternative modes remain separate experiments.

The anchor comes from [jazivxt's public dataset](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline). Its hash matches the catalog entry for submission **56197564**, reported as **0.94649**. The secondary model is [yekenot's RealMLP, version 3](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch).

## 1. Attach data

Attach the S6E9 competition data. Upload `s6e9_lexsort_trial_inputs.zip` as a private Kaggle dataset and attach it too. CPU is sufficient; Internet can be off.

The final cell always writes **`/kaggle/working/submission.csv`**.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

# Try the published notebook output first.
# Other choices: "anchor" or "anchor_lexsort".
MODE = "public_notebook"

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Load the exact files and match IDs

File hashes pin this experiment to the downloaded predictions. ID matching prevents accidental blending of different row orders.

In [ ]:
EXPECTED_HASHES = {'public_lexsort.csv': '57e8ee944c1ac39c0483bb977f1f9c67af419a105ca00942b953569802530ada', 'anchor_56197564.csv': '3a1ce448f12b6a6d42ff8dec7cc53a9b5e413b2327b8fe59f5097c688c513b85', 'realmlp_v3.csv': 'ea5cb35eb5b3afdd16d3aa3f0d74227fbbd6fba816cc3783ca6d3c49929f2f77'}

def find_unique(name):
    matches = list(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f"Expected one {name}; found {len(matches)}. Check attached datasets.")
    return matches[0]

sample = pd.read_csv(find_unique("sample_submission.csv"))
assert sample.id.is_unique and len(sample) == 286571

def load_predictions(name):
    path = find_unique(name)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == EXPECTED_HASHES[name], "Input version differs"
    frame = pd.read_csv(path, float_precision="round_trip")
    assert list(frame) == ["id", "Will_Buy_EV"]
    assert frame.id.is_unique and set(frame.id) == set(sample.id)
    values = frame.set_index("id").loc[sample.id, "Will_Buy_EV"].to_numpy()
    assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
    return values

public = load_predictions("public_lexsort.csv")
anchor = load_predictions("anchor_56197564.csv")
neural = load_predictions("realmlp_v3.csv")

pd.DataFrame({"source": ["public", "anchor", "RealMLP"],
              "repeated_values": [len(p) - len(np.unique(p)) for p in [public, anchor, neural]]})

## 3. Choose one controlled trial

- **public_notebook:** use the public notebook's already-generated output unchanged.
- **anchor:** use the catalog-verified anchor alone.
- **anchor_lexsort:** preserve distinct anchor rankings and use RealMLP to order exact ties. This is a new unscored experiment, not a reproduction of the author's complete 90/6/4 blend.

Tie-breaking can improve or worsen AUC. No matching OOF library was retrieved for the complete public blend, so we cannot optimize its weights locally. The optional experiment adds no hard boundary overrides. The downloaded public output already contains its author's transformations.

In [ ]:
def break_ties(primary, secondary):
    order = np.lexsort((secondary, primary))
    ranks = np.empty(len(order), dtype=np.int64)
    ranks[order] = np.arange(len(order))
    # If both sources tie, lexsort uses their input order; this adds no learned signal.
    return (ranks + 0.5) / len(order)

choices = {
    "public_notebook": public,
    "anchor": anchor,
    "anchor_lexsort": break_ties(anchor, neural),
}
if MODE not in choices:
    raise ValueError("Choose public_notebook, anchor or anchor_lexsort")
prediction = choices[MODE]

## 4. Export submission.csv

Save a version with **Run All**. Open **Output**, download `submission.csv`, and submit it to the competition. Record the actual returned LB before changing the mode. Keep your own 0.94639 model submission separately.

In [ ]:
submission = sample[["id"]].copy()
submission["Will_Buy_EV"] = prediction
assert submission.id.equals(sample.id)
assert np.isfinite(prediction).all()
assert submission.Will_Buy_EV.between(0, 1).all()
path = OUTPUT_ROOT / "submission.csv"
submission.to_csv(path, index=False, float_format="%.17g")
print(f"Created {path}: {len(submission):,} rows | mode={MODE}")
print("Public LB: 0.94650 (user-reported for this public output)." if MODE == "public_notebook" else "Public LB for this alternative: not measured in this trial.")
display(submission.head())